In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import anndata as ad
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.sparse import issparse

In [ ]:
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)

In [ ]:
SAMPLE_NAME = "P22"
SAMPLE_DIR  = "sample_P22"

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/spatial/reg_visium/{SAMPLE_DIR}/output/filtered/{SAMPLE_NAME}_filtered.h5ad')

In [ ]:
N_TOP_GENES      = 10000
N_NEIGHBORS      = 15
N_SPATIAL_NEIGHS = 5

In [ ]:
# ★ Save raw counts for cell2location
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["log1p_norm"] = adata.X.copy()
adata.raw = adata

sc.pp.highly_variable_genes(
    adata, n_top_genes=N_TOP_GENES, flavor="seurat_v3",
    layer="counts", subset=False,
)
print(f"HVGs: {adata.n_vars:,} genes retained")

In [ ]:
# highly variable genes
adata_hvg = adata[:, adata.var["highly_variable"]].copy()

In [ ]:
N_PCS = 50

In [ ]:
# PCA
sc.pp.pca(adata_hvg, n_comps=N_PCS)

In [ ]:
sc.pl.pca_variance_ratio(adata_hvg, log = True, n_pcs = N_PCS)

In [ ]:
# adding information back to the main adata to preserve all the genes.
adata.obsm["X_pca"] = adata_hvg.obsm["X_pca"]
adata.uns["pca"]    = adata_hvg.uns["pca"]
# adata.varm["PCs"] = adata_hvg.varm["PCs"]
del adata_hvg

In [ ]:
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=30)
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata, key_added="leiden", resolution=0.3)

In [ ]:
sc.pl.umap(adata, color = ['leiden'], size=10)

In [ ]:
coords = adata.obsm["spatial"]
pad = 200  # pixel padding around the capture area

x0 = int(coords[:, 0].min()) - pad
x1 = int(coords[:, 0].max()) + pad
y0 = int(coords[:, 1].min()) - pad
y1 = int(coords[:, 1].max()) + pad

sq.pl.spatial_scatter(
    adata,
    color="leiden",
    # library_id=SAMPLE_NAME,   
    size=1,
    title="Leiden clusters on tissue",
    figsize=(8, 8),
    crop_coord=(x0, y0, x1, y1),  # squidpy: (xmin, ymin, xmax, ymax)
)

In [ ]:
sc.pl.spatial(adata, color=None, img_key="hires",
              size=2.0, cmap="magma")

In [ ]:
sc.pl.umap(adata, color = ['MYB'], size=20)
sc.pl.umap(adata, color = ['NOTCH1'], size=20)

In [ ]:
def fix_arrow_strings(df):
    df.index = df.index.astype(object)
    for col in df.columns:
        if isinstance(df[col].dtype, pd.CategoricalDtype):
            # Fix Arrow-backed categories
            cats = df[col].cat.categories
            if pd.api.types.is_extension_array_dtype(cats):
                df[col] = df[col].cat.rename_categories(cats.astype(object))
        elif pd.api.types.is_extension_array_dtype(df[col]):
            df[col] = df[col].astype(object)

fix_arrow_strings(adata.obs)
fix_arrow_strings(adata.var)

if adata.raw is not None:
    fix_arrow_strings(adata.raw._var)

adata.write_h5ad(
    f'{ACC_DATA_ROOT}/spatial/reg_visium/{SAMPLE_DIR}/output/filtered/{SAMPLE_NAME}_clustered.h5ad',
    compression='lzf'
)
